# rag-starter: Your First RAG Pipeline (Local Edition)

This notebook walks you through building a RAG pipeline using **Ollama** running
locally on your machine. Everything stays on your computer — no API keys, no cloud.

**Prerequisites:**
- Python 3.10+
- Ollama installed and running (`ollama serve`)
- Models pulled: `ollama pull llama3.1:8b && ollama pull nomic-embed-text`
- Dependencies installed: `pip install -r quickstart/requirements.txt`

Run `bash quickstart/install.sh` (or `quickstart\install.ps1` on Windows) to set
everything up automatically.

## Step 0: Check prerequisites

Let's verify Ollama is running and the required models are available.

In [ ]:
import ollama

try:
    models = ollama.list()
    available = [m.model for m in models.models] if hasattr(models, 'models') else []
    print("Ollama is running!")
    print(f"Available models: {', '.join(available) if available else 'none'}")

    required = ["llama3.1:8b", "nomic-embed-text"]
    for model in required:
        found = any(model in m for m in available)
        status = '\u2713' if found else '\u2717 (run: ollama pull ' + model + ')'
        print(f"  {status} {model}")
except Exception as e:
    print(f"Could not connect to Ollama: {e}")
    print("Start it with: ollama serve")

## Step 1: Add project to path

So we can import from the `src/` module.

In [ ]:
import os
import sys

# Navigate to the project root (parent of notebooks/)
project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, project_root)
print(f"Project root: {project_root}")

## Step 2: Load documents

We'll load the sample documents that come with rag-starter.

In [ ]:
from src.loader import load_folder

sample_dir = os.path.join(project_root, "sample-docs")
docs = load_folder(sample_dir, verbose=True)
print(f"\nLoaded {len(docs)} documents")
for doc in docs:
    print(f"  - {doc.metadata['source']} ({len(doc.content)} chars)")

## Step 3: Chunk the documents

Split each document into smaller overlapping pieces for more precise retrieval.

In [ ]:
from src.chunker import chunk_documents

chunk_texts, chunk_metadatas = chunk_documents(docs, chunk_size=500, chunk_overlap=50)
print(f"Created {len(chunk_texts)} chunks")
print(f"\nExample chunk (first 200 chars):\n{chunk_texts[0][:200]}...")

## Step 4: Generate embeddings

Convert each chunk into a list of 768 numbers that capture its meaning.

In [ ]:
from src.embedder import embed_texts

embeddings = embed_texts(chunk_texts)
print(f"Generated {len(embeddings)} embeddings, each with {len(embeddings[0])} dimensions")

## Step 5: Store in ChromaDB

Save the embeddings in a local vector database so we can search them later.

In [ ]:
import tempfile

from src.store import add_documents, create_store

# Use a temp directory so we don't pollute the project
db_path = tempfile.mkdtemp(prefix="rag_notebook_")
client, collection = create_store(path=db_path)
add_documents(collection, chunk_texts, embeddings, chunk_metadatas)
print(f"Stored {collection.count()} chunks in ChromaDB at {db_path}")

## Step 6: Query with RAG

Now the full pipeline: embed the question, retrieve relevant chunks, and generate
an answer grounded in your documents.

In [ ]:
from src.embedder import embed_query
from src.generator import generate_answer
from src.store import query_store


def ask(question: str, top_k: int = 3):
    """Full RAG pipeline: embed, retrieve, generate."""
    query_vec = embed_query(question)
    results = query_store(collection, query_vec, top_k=top_k)
    context_chunks = [r["content"] for r in results]
    answer = generate_answer(question, context_chunks)

    print(f"Question: {question}\n")
    print(f"Answer: {answer}\n")
    sources = set(r["metadata"].get("source", "?") for r in results)
    print(f"Sources: {', '.join(sources)}")
    print("-" * 60)

ask("What are the best practices for async communication in remote teams?")
ask("How does supervised learning differ from unsupervised learning?")
ask("What neighbourhoods should I visit in Berlin?")

## Try your own!

Type any question about the sample documents.

In [ ]:
your_question = input("Your question: ")
if your_question.strip():
    ask(your_question)

## Cleanup

Remove the temporary database.

In [ ]:
import shutil

shutil.rmtree(db_path, ignore_errors=True)
print("Cleaned up temporary database.")